# Day 2 — Tools & Function Calling

---

Yesterday we hand-parsed `Action: calc[...]` from LLM output with a regex. That works, but modern LLMs support a cleaner API called **function calling** (a.k.a. **tool use**) where the model returns *structured JSON* instead of a plain string.

Today you'll:

1. Use OpenAI-style function calling — no regex, no hallucinated tool names
2. Add three real tools: **web search** (Tavily), **HTTP fetch**, and a **safe calculator**
3. Build a multi-tool agent that picks the right tool for the job


## 1. Why function calling beats regex parsing

The regex approach from Day 1 has real failure modes:
- The model writes `Action: calculate[...]` instead of `calc[...]` — regex misses.
- The model writes `Action:calc[2+2]` (no space) — regex misses.
- The model wraps in backticks — regex misses.

**Function calling** removes the parsing entirely. You give the model a JSON schema of each tool. The API returns a structured `{"name": "calc", "arguments": {"expr": "2+2"}}`. No regex, no fuzzy matching. This is what Claude, GPT-4, and Llama 3 all support now.


## 2. Setup


In [ ]:
!pip install together tavily-python httpx python-dotenv --quiet

In [ ]:
import os, json
from dotenv import load_dotenv
load_dotenv()
from together import Together
llm = Together()


## 3. Define tool schemas

Each tool = a JSON schema describing its name, purpose, and parameters. The LLM reads these to pick the right tool.


In [ ]:
TOOL_SCHEMAS = [
    {
        "type": "function",
        "function": {
            "name": "web_search",
            "description": "Search the web for recent information. Returns short snippets.",
            "parameters": {
                "type": "object",
                "properties": {
                    "query": {"type": "string", "description": "The search query."}
                },
                "required": ["query"],
            },
        },
    },
    {
        "type": "function",
        "function": {
            "name": "fetch_url",
            "description": "Fetch the plain-text content of a URL.",
            "parameters": {
                "type": "object",
                "properties": {"url": {"type": "string"}},
                "required": ["url"],
            },
        },
    },
    {
        "type": "function",
        "function": {
            "name": "calc",
            "description": "Evaluate a simple math expression like '(29*12)+100'.",
            "parameters": {
                "type": "object",
                "properties": {"expr": {"type": "string"}},
                "required": ["expr"],
            },
        },
    },
]


**Writing good tool descriptions is the #1 lever for agent quality.** Be specific: *what* it does, *when* to use it, *what* it returns. Vague descriptions → the model guesses wrong.


## 4. Implement the tools


In [ ]:
# --- Safe calculator (same as Day 1) ---
import ast, operator
_OPS = {ast.Add: operator.add, ast.Sub: operator.sub,
        ast.Mult: operator.mul, ast.Div: operator.truediv,
        ast.Pow: operator.pow, ast.USub: operator.neg}
def _eval(n):
    if isinstance(n, ast.Num): return n.n
    if isinstance(n, ast.BinOp): return _OPS[type(n.op)](_eval(n.left), _eval(n.right))
    if isinstance(n, ast.UnaryOp): return _OPS[type(n.op)](_eval(n.operand))
    raise ValueError("bad expr")

def calc(expr: str) -> str:
    try: return str(_eval(ast.parse(expr, mode="eval").body))
    except Exception as e: return f"error: {e}"


# --- Web search (Tavily) ---
def web_search(query: str) -> str:
    if not os.getenv("TAVILY_API_KEY"):
        # Fallback: fake result so the demo still runs
        return f"[MOCK] top result for '{query}': ..."
    from tavily import TavilyClient
    tv = TavilyClient(api_key=os.getenv("TAVILY_API_KEY"))
    r = tv.search(query, max_results=3)
    return "\n".join(f"- {x['title']}: {x['content'][:200]}" for x in r["results"])


# --- HTTP fetch ---
import httpx
def fetch_url(url: str) -> str:
    try:
        r = httpx.get(url, timeout=10, follow_redirects=True)
        r.raise_for_status()
        # Return a truncated version - agents drown in raw HTML
        text = r.text
        return text[:2000] + ("... [truncated]" if len(text) > 2000 else "")
    except Exception as e:
        return f"error: {e}"


TOOLS = {"web_search": web_search, "fetch_url": fetch_url, "calc": calc}


**Two tool-writing rules already visible above:**

1. **Truncate outputs.** LLMs can't usefully read 100 KB of HTML. Cap at ~2 KB and let the agent decide if it wants more.
2. **Always return a string.** Not a dict, not an exception — a string. The agent will paste it into context; anything else breaks the loop.


## 5. The function-calling loop


In [ ]:
MODEL = "openai/gpt-oss-20b"

def agent(question: str, max_steps: int = 6) -> str:
    messages = [
        {"role": "system", "content":
            "You are a helpful research assistant. Use tools when they are useful. "
            "When you have the final answer, respond in plain text without tool calls."},
        {"role": "user", "content": question},
    ]

    for step in range(max_steps):
        resp = llm.chat.completions.create(
            model=MODEL,
            messages=messages,
            tools=TOOL_SCHEMAS,
            tool_choice="auto",
            temperature=0.0,
        )
        msg = resp.choices[0].message

        # If the model asked for tools, run them and loop
        if msg.tool_calls:
            messages.append({"role": "assistant",
                             "content": msg.content or "",
                             "tool_calls": [tc.model_dump() for tc in msg.tool_calls]})
            for tc in msg.tool_calls:
                name = tc.function.name
                args = json.loads(tc.function.arguments)
                arg_value = next(iter(args.values()))    # our tools take one arg each
                obs = TOOLS[name](arg_value) if name in TOOLS else f"unknown tool {name}"
                print(f"\n--- step {step+1}: {name}({arg_value!r}) ---\n{obs[:400]}")
                messages.append({
                    "role": "tool", "tool_call_id": tc.id,
                    "name": name, "content": obs,
                })
            continue

        # No tool calls => final answer
        return msg.content or "(empty)"

    return "(max steps reached)"


print(agent("What is the current price of Bitcoin in USD, roughly?"))


**Loop shape:** send messages → model may respond with `tool_calls` → for each call, run the tool and append a `{"role": "tool", ...}` message → send back to model → repeat until the model returns plain text (no tool calls).

This is *literally* how Claude Code, Cursor, and every commercial agent talks to its LLM. Same protocol.


## 6. OpenAI-style vs Together vs Anthropic

Good news: **the tool-calling shape is nearly identical across providers**. Same JSON schemas, same `tool_calls` in the response, same `tool` role for observations. Once you learn one you can switch in an afternoon.

Minor variations:
- OpenAI historically had `functions=` (deprecated) then `tools=`. Use `tools=` today.
- Anthropic's SDK uses `input_schema` instead of `parameters`. Semantically identical.
- Some open models are worse at picking the right tool. Llama 3.3 70B is very good; smaller models drop off.


## 7. Failure modes to watch for

Even with function calling, agents misbehave. Common issues:

- **Loops.** Model calls the same tool with the same args over and over. Fix: track call history, break on repeat (Day 5).
- **Wrong tool.** Description was too vague. Fix: rewrite the description with an example.
- **Bad arguments.** Model passes `"twenty nine"` when the schema wants a number. Fix: `type: number` + example in description.
- **Hallucinated tools.** Rare with function calling, common with regex parsing. Fix: use function calling (like today).


## Recap

- **Function calling** = structured JSON tool requests. No regex, no fragile parsing.
- Every tool = **schema (name + description + params) + Python function**.
- **Truncate tool outputs** and always **return strings**.
- Loop shape: send messages → if `tool_calls`, run + append → else, return final answer.
- Same protocol across OpenAI, Together, Anthropic — learn once.
- **Next class:** LangGraph — build agent workflows that are stateful, resumable, and easy to reason about.
